# Actividad 4.5.3

## Auditoria interna de satisfaccion de usuario mediante NLP

Este cuaderno reutiliza el enfoque de las actividades `4.5.1` y `4.5.2` 

Objetivos del notebook:
- Entrenar un clasificador binario de reseñas positivas y negativas.
- Analizar un conjunto de reseñas de un producto de Amazon.
- Obtener automaticamente puntos fuertes y puntos debiles.
- Procesar una reseña nueva y actualizar el informe.


In [15]:
# Instalando dependencias
%pip install numpy tensorflow scikit-learn


Note: you may need to restart the kernel to use updated packages.


## 1. Datos del producto

Producto analizado: **JBL Tune 510BT**

Para resolver la actividad se usa:
- Un pequeño conjunto etiquetado de entrenamiento inspirado en los ejemplos de clase de `4.5.2`.
- Un lote de 20 reseñas del producto para clasificar y analizar.

Las estrellas se usan solo como referencia para comprobar si la prediccion del modelo parece coherente con la opinion general de cada reseña.


In [ ]:
import re
import numpy as np
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tf.keras.utils.set_random_seed(42)

# Nombre del producto analizado
producto = "JBL Tune 510BT"

# Reseñas positivas
frases_positivas = [
    "El sonido es excelente y muy claro",
    "La bateria dura muchisimo",
    "Son comodos incluso despues de horas",
    "La conexion bluetooth es rapida y estable",
    "Muy buena relacion calidad precio",
    "Los graves suenan potentes",
    "Se sincronizan en segundos con el movil",
    "El volumen es alto y limpio",
    "Me encanta el diseno y el acabado",
    "La carga es rapida y practica",
    "Son ligeros y faciles de llevar",
    "El microfono funciona bien en llamadas",
    "Las almohadillas son suaves y comodas",
    "Aislan bastante bien el ruido exterior",
    "Los botones responden perfectamente",
    "Excelente autonomia para usar todo el dia",
    "El audio se escucha equilibrado",
    "Buena compra, cumplen de sobra",
    "El alcance del bluetooth es muy bueno",
    "Se ven resistentes y bien construidos"
]

# Reseñas negativas
frases_negativas = [
    "Aprietan demasiado y hacen dano",
    "El microfono se oye fatal",
    "La bateria dura muy poco",
    "Se desconectan todo el tiempo",
    "El plastico parece barato y fragil",
    "El volumen es bajo para lo que cuestan",
    "Las almohadillas dan mucho calor",
    "El sonido sale distorsionado",
    "No aislan nada del ruido exterior",
    "Son incomodos despues de media hora",
    "La conexion bluetooth falla a ratos",
    "Los botones dejan de responder",
    "Tardan mucho en cargar",
    "Mala relacion calidad precio",
    "Se sienten endebles y mal acabados",
    "El audio tiene poco cuerpo",
    "No cumplen lo que prometen",
    "Muy decepcionado con la compra",
    "La diadema molesta bastante",
    "Las llamadas se escuchan mal"
]

# Combinamos las frases y creamos etiquetas (1 para positivo y 0 para negativo)
frases_entrenamiento = frases_positivas + frases_negativas
etiquetas = np.array([1] * len(frases_positivas) + [0] * len(frases_negativas))

reseñas = [
    {"estrellas": 5, "texto": "El sonido es potente y la bateria me dura casi toda la semana."},
    {"estrellas": 5, "texto": "Se conectan al movil en segundos y no he tenido cortes."},
    {"estrellas": 4, "texto": "Muy comodos para teletrabajar, apenas pesan y no cansan."},
    {"estrellas": 5, "texto": "Buena relacion calidad precio, por este coste suenan genial."},
    {"estrellas": 4, "texto": "La carga rapida cumple y en una hora los tengo listos."},
    {"estrellas": 5, "texto": "Los uso para estudiar y aislan bastante bien el ruido exterior."},
    {"estrellas": 4, "texto": "El bluetooth alcanza varias habitaciones y sigue estable."},
    {"estrellas": 5, "texto": "Los graves estan muy bien para escuchar musica urbana."},
    {"estrellas": 4, "texto": "Se pliegan facil y caben en la mochila sin problema."},
    {"estrellas": 5, "texto": "La bateria es una pasada, los cargue el lunes y siguen con carga."},
    {"estrellas": 4, "texto": "Los botones fisicos responden mejor de lo que esperaba."},
    {"estrellas": 5, "texto": "Para videollamadas se escuchan nitidos y el volumen es alto."},
    {"estrellas": 4, "texto": "La diadema parece resistente y el acabado se ve bonito."},
    {"estrellas": 5, "texto": "Muy buena compra, ligeros, comodos y con sonido equilibrado."},
    {"estrellas": 2, "texto": "Aprietan demasiado despues de media hora y molestan en las orejas."},
    {"estrellas": 1, "texto": "El microfono recoge mi voz muy baja y en las llamadas se quejan."},
    {"estrellas": 2, "texto": "El plastico se siente barato y cruje al abrirlos."},
    {"estrellas": 1, "texto": "La conexion bluetooth falla a ratos y se desconectan solos."},
    {"estrellas": 2, "texto": "Esperaba mas volumen, al maximo se quedan cortos."},
    {"estrellas": 1, "texto": "Las almohadillas dan calor enseguida y resultan incomodas en verano."},
    {"estrellas": 2, "texto": "No aislan tanto como dicen, se escucha todo en el autobus."},
    {"estrellas": 1, "texto": "A los dos meses la bateria empezo a durar mucho menos."},
    {"estrellas": 2, "texto": "El boton de encendido va duro y a veces no responde."},
    {"estrellas": 1, "texto": "Por el precio no compensan, hay opciones mejores y mas comodas."}
]

# Filtramos las palabras vacias para ello usamos NLTK
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('spanish'))


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ciabd10/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 2. Preprocesamiento del texto

Igual que en `4.5.2`, convertimos las frases en secuencias numericas y aplicamos `padding` para que todas tengan la misma longitud antes de entrar en la red neuronal.


In [17]:
# Definimos una serie de parámetros para el tokenizador y homogeneización
vocab_size = 1000
max_length = 20
oov_tok = "<OOV>"
padding_type = "post"
trunc_type = "post"

# Aplicando el tokenizador
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(frases_entrenamiento)

# Aplicando el tokenizador a las frases de entrenamiento y homogeneizando la longitud
sequences = tokenizer.texts_to_sequences(frases_entrenamiento)
padded = pad_sequences(
    sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type,
)

print("Frases de entrenamiento:", len(frases_entrenamiento))
print("Tamaño del vocabulario aprendido:", len(tokenizer.word_index))
print("Forma del tensor de entrada:", padded.shape)


Frases de entrenamiento: 40
Tamaño del vocabulario aprendido: 126
Forma del tensor de entrada: (40, 20)


## 3. Modelo LSTM para clasificacion de sentimiento

En este bloque reutilizamos la arquitectura pedida en el enunciado: `Embedding + Bidirectional LSTM + Dense`.


In [18]:
# Creando la red neuronal para procesar los tipos de reseñas
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 32, input_length=max_length),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.summary()
history = model.fit(padded, etiquetas, epochs=35, verbose=0)
loss, accuracy = model.evaluate(padded, etiquetas, verbose=0)

print(f"Accuracy sobre el conjunto de entrenamiento: {accuracy:.2%}")


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Accuracy sobre el conjunto de entrenamiento: 95.00%


## 4. Clasificacion de las reseñas del producto

Ahora aplicamos la red a las reseñas recopiladas y las separamos en dos grupos: usuarios satisfechos e insatisfechos.


In [19]:
# Creación de una función para clasificar nuevas reseñas y comparar con las estrellas asignadas
def clasificar_resena(texto):
    secuencia = tokenizer.texts_to_sequences([texto])
    relleno = pad_sequences(
        secuencia,
        maxlen=max_length,
        padding=padding_type,
        truncating=trunc_type,
    )
    probabilidad = float(model.predict(relleno, verbose=0)[0][0])
    sentimiento = "Positiva" if probabilidad >= 0.5 else "Negativa"
    return sentimiento, probabilidad

reseñas_clasificadas = []

# Clasificando cada reseña y comparandola con la cantidad de estrellas
for item in reseñas:
    sentimiento, probabilidad = clasificar_resena(item["texto"])
    pred_bin = 1 if sentimiento == "Positiva" else 0
    real_bin = 1 if item["estrellas"] >= 4 else 0
    reseñas_clasificadas.append({
        "estrellas": item["estrellas"],
        "texto": item["texto"],
        "sentimiento": sentimiento,
        "probabilidad": probabilidad,
        "acierto": pred_bin == real_bin,
    })

positivas = [r for r in reseñas_clasificadas if r["sentimiento"] == "Positiva"]
negativas = [r for r in reseñas_clasificadas if r["sentimiento"] == "Negativa"]

porcentaje_positivas = len(positivas) / len(reseñas_clasificadas) * 100
porcentaje_negativas = len(negativas) / len(reseñas_clasificadas) * 100
porcentaje_acierto = sum(r["acierto"] for r in reseñas_clasificadas) / len(reseñas_clasificadas) * 100

print("Producto analizado:", producto)
print("Reseñas analizadas:", len(reseñas_clasificadas))
print("Porcentaje de reseñas positivas:", f"{porcentaje_positivas:.2f}%")
print("Porcentaje de reseñas negativas:", f"{porcentaje_negativas:.2f}%")
print("Coincidencia del modelo con las estrellas:", f"{porcentaje_acierto:.2f}%")
print()

for r in reseñas_clasificadas:
    print(f"[{r['estrellas']} estrellas] {r['sentimiento']} ({r['probabilidad']:.2f})")
    print(r["texto"])
    print()


Producto analizado: JBL Tune 510BT
Reseñas analizadas: 24
Porcentaje de reseñas positivas: 58.33%
Porcentaje de reseñas negativas: 41.67%
Coincidencia del modelo con las estrellas: 75.00%

[5 estrellas] Positiva (0.92)
El sonido es potente y la bateria me dura casi toda la semana.

[5 estrellas] Positiva (0.89)
Se conectan al movil en segundos y no he tenido cortes.

[4 estrellas] Negativa (0.18)
Muy comodos para teletrabajar, apenas pesan y no cansan.

[5 estrellas] Positiva (0.94)
Buena relacion calidad precio, por este coste suenan genial.

[4 estrellas] Positiva (0.92)
La carga rapida cumple y en una hora los tengo listos.

[5 estrellas] Positiva (0.93)
Los uso para estudiar y aislan bastante bien el ruido exterior.

[4 estrellas] Positiva (0.90)
El bluetooth alcanza varias habitaciones y sigue estable.

[5 estrellas] Positiva (0.94)
Los graves estan muy bien para escuchar musica urbana.

[4 estrellas] Positiva (0.61)
Se pliegan facil y caben en la mochila sin problema.

[5 estrell

## 5. Extraccion automatica de puntos fuertes y puntos debiles

Aqui reaprovechamos la idea de `4.5.1`: usamos `TF-IDF` para encontrar los conceptos mas representativos dentro de cada grupo de resenas.


In [ ]:
def limpiar_texto(texto):
    # Convertimos el texto a minúsculas y luego eliminamos caracteres que no son 
    # alfanuméricos ni espacios, solo se dejan acentos y caracteres comunes
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    palabras = [palabra for palabra in texto.split() if palabra not in stop_words]
    return " ".join(palabras)

def extraer_puntos_clave(lista_textos, n=5):
    if not lista_textos: 
        return []
    
    # Limpiando los textos usando la funcion global
    docs_limpios = [limpiar_texto(t) for t in lista_textos]
    
    # Configurando TF-IDF para detectar palabras y frases
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    
    # Calculamos la importancia de cada término
    matriz = vectorizer.fit_transform(docs_limpios)
    puntuaciones = np.asarray(matriz.sum(axis=0)).ravel()
    terminos = vectorizer.get_feature_names_out()
    
    # Obtenemos los 'n' términos con mayor puntuación
    indices_top = puntuaciones.argsort()[::-1][:n]
    return [terminos[i] for i in indices_top]

# Ejecución y resultados
puntos_fuertes = extraer_puntos_clave([r["texto"] for r in positivas], n=5)
puntos_debiles = extraer_puntos_clave([r["texto"] for r in negativas], n=5)

print(f"PUNTOS FUERTES: {', '.join(puntos_fuertes)}")
print(f"PUNTOS DÉBILES: {', '.join(puntos_debiles)}")

PUNTOS FUERTES: aislan, carga, sonido, buena, bateria
PUNTOS DÉBILES: esperaba, plastico barato, cruje abrirlos, cruje, barato


## 6. Logica de negocio: procesar una resena nueva

El siguiente bloque permite pasar una resena nueva al sistema, clasificarla automaticamente y actualizar los puntos fuertes o debiles detectados.


In [ ]:
def procesar_nueva_reseña(texto):
    # Clasifica una reseña
    sentimiento, prob = clasificar_resena(texto)

    # Función para obtener el texto de una reseña
    def obtener_texto_reseña(reseña):
        return reseña["texto"] if isinstance(reseña, dict) else reseña
    
    if sentimiento == "Positiva":
        positivas.append({"texto": texto}) 
    else:
        negativas.append({"texto": texto})

    return {
        "sentimiento": sentimiento,
        "probabilidad": prob,
        "fuertes": extraer_puntos_clave([obtener_texto_reseña(reseña) for reseña in positivas]),
        "debiles": extraer_puntos_clave([obtener_texto_reseña(reseña) for reseña in negativas])
    }

# Prueba
nueva_reseña = "El sonido es bueno, pero el bluetooth se corta constantemente y el microfono es flojo."
res = procesar_nueva_reseña(nueva_reseña)

print(f"--- ANÁLISIS DE NUEVA RESEÑA ---")
print(f"Texto: {nueva_reseña}")
print(f"Clasificación: {res['sentimiento']} ({res['probabilidad']:.2f})")
print(f"\nNuevos Puntos Fuertes: {res['fuertes']}")
print(f"Nuevos Puntos Débiles: {res['debiles']}")

--- ANÁLISIS DE NUEVA RESEÑA ---
Texto: El sonido es bueno, pero el bluetooth se corta constantemente y el microfono es flojo.
Clasificación: Positiva (0.88)

Nuevos Puntos Fuertes: ['sonido', 'aislan', 'carga', 'buena', 'bateria']
Nuevos Puntos Débiles: ['esperaba', 'plastico barato', 'cruje abrirlos', 'cruje', 'barato']


## 7. Conclusión automática

Cerramos el cuaderno con una conclusion breve sobre si el modelo parece clasificar con logica las resenas del producto.


In [22]:
# Condicional para evaluar el resultado del modelo y hacer una conclusión
if porcentaje_acierto >= 80:
    conclusion = "El modelo parece clasificar con bastante coherencia las reseñas del producto. La separacion entre comentarios positivos y negativos coincide en gran medida con la valoracion por estrellas."
elif porcentaje_acierto >= 65:
    conclusion = "El modelo ofrece resultados razonables, aunque todavia hay margen de mejora. Con mas datos reales de entrenamiento podria afinar mejor los casos ambiguos."
else:
    conclusion = "El modelo necesita mas ejemplos de entrenamiento porque la coincidencia con las estrellas aun es baja. La arquitectura es correcta, pero el conjunto de datos deberia ampliarse."

print("INFORME FINAL")
print(f"Producto analizado: {producto}")
print(f"Reseñas positivas: {porcentaje_positivas:.2f}%")
print(f"Reseñas negativas: {porcentaje_negativas:.2f}%")
print("Puntos fuertes extraidos:", puntos_fuertes)
print("Puntos debiles extraidos:", puntos_debiles)
print("Conclusion:")
print(conclusion)


INFORME FINAL
Producto analizado: JBL Tune 510BT
Reseñas positivas: 58.33%
Reseñas negativas: 41.67%
Puntos fuertes extraidos: ['aislan', 'carga', 'sonido', 'buena', 'bateria']
Puntos debiles extraidos: ['esperaba', 'plastico barato', 'cruje abrirlos', 'cruje', 'barato']
Conclusion:
El modelo ofrece resultados razonables, aunque todavia hay margen de mejora. Con mas datos reales de entrenamiento podria afinar mejor los casos ambiguos.
